# Africa's Growing Presence at the World Cup
### SWD July 2026 Challenge — Blessing Obasi-Uzoma

**Story:** From 0 seats to 9 — how Africa's representation at the Men's World Cup has grown since 1930, and what the 2026 expansion finally means for the continent.

**Data Source:** Fjelstul World Cup Database (github.com/jfjelstul/worldcup)

In [ ]:
# Import libraries
# pandas handles all our data manipulation
import pandas as pd

In [ ]:
# Load the three source files
# Use r'' (raw string) to handle Windows backslashes in file paths
qualified_teams = pd.read_csv(r'qualified_teams.csv')
tournaments = pd.read_csv(r'tournaments.csv')
tournament_standings = pd.read_csv(r'tournament_standings.csv')

print('qualified_teams:', qualified_teams.shape)
print('tournaments:', tournaments.shape)
print('tournament_standings:', tournament_standings.shape)

## Step 1: Filter for Men's World Cup Only

The dataset includes both Men's and Women's World Cups.
We filter using `.str.contains('Men')` which keeps only rows
where tournament_name contains the word 'Men'.

In [ ]:
# Filter for Men's World Cup only
# .copy() creates an independent DataFrame so we don't get SettingWithCopyWarning
men_wc = qualified_teams[
    qualified_teams['tournament_name'].str.contains("Men's")
].copy()

print('Men World Cup records:', men_wc.shape)
print('Unique tournaments:', men_wc['tournament_name'].nunique())

## Step 2: Define African Teams

There is no confederation column in this dataset.
We manually define the list of African nations that have
appeared in the Men's World Cup. These belong to CAF
(Confederation of African Football).

In [ ]:
# List of African nations that have appeared at the Men's World Cup
african_teams = [
    'Algeria', 'Angola', 'Cameroon', 'Egypt', 'Ghana',
    'Ivory Coast', 'Morocco', 'Nigeria', 'Senegal',
    'South Africa', 'Togo', 'Tunisia', 'Zaire',
    'Equatorial Guinea'
]

# Filter for African teams only using .isin()
# .isin() checks if each value exists in our list
africa_wc = men_wc[men_wc['team_name'].isin(african_teams)].copy()

print('African team appearances:', africa_wc.shape)
print('African teams found:', sorted(africa_wc['team_name'].unique().tolist()))

## Step 3: Add Tournament Year

The qualified_teams file has tournament_id but not the year.
We merge with the tournaments file to get the year column.
This is like a JOIN in SQL.

In [ ]:
# Get year from tournaments file — Men's only
men_tournaments = tournaments[
    tournaments['tournament_name'].str.contains("Men's")
][['tournament_id', 'year']].copy()

# Merge to add year column to Africa data
# how='left' keeps all African records even if no match found
africa_wc = africa_wc.merge(men_tournaments, on='tournament_id', how='left')

print(africa_wc[['year', 'team_name', 'performance']].head(10).to_string())

## Step 4: Rank Performance Stages

The 'performance' column has text values like 'group stage',
'round of 16', 'quarter-finals'. We need numbers to find
the BEST performance each year.

We create a ranking dictionary and map it to a new column.

In [ ]:
# Map performance text to numbers
# Higher number = better performance
perf_rank = {
    'group stage': 1,
    'second group stage': 2,
    'final round': 2,
    'round of 16': 3,
    'quarter-final': 4,
    'quarter-finals': 4,
    'third-place match': 5,
    'semi-finals': 6,
    'final': 7
}

# .map() replaces each value using the dictionary
africa_wc['perf_rank'] = africa_wc['performance'].map(perf_rank)

print(africa_wc[['year', 'team_name', 'performance', 'perf_rank']].head(10).to_string())

## Step 5: Aggregate by Year

We group by year and calculate:
- How many African teams qualified that year (count)
- The best performance any African team achieved (max rank)
- Which team achieved that best performance

In [ ]:
# Aggregate per year
yearly = africa_wc.groupby('year').agg(
    African_Teams=('team_name', 'count'),
    Best_Performance=('perf_rank', 'max'),
    Best_Team=('team_name', lambda x: x[africa_wc.loc[x.index, 'perf_rank'].idxmax()])
).reset_index()

# Map rank back to readable label
rank_to_label = {
    1: 'Group stage', 2: 'Second group stage', 3: 'Round of 16',
    4: 'Quarter-finals', 5: 'Third place', 6: 'Semi-finals', 7: 'Final'
}
yearly['Best_Performance_Label'] = yearly['Best_Performance'].map(rank_to_label)

print(yearly.to_string())

## Step 6: Add Missing Years and 2026

Years where Africa had 0 teams don't appear in our data.
We add them manually so the timeline is complete.
We also manually add 2026 since it's not in the database yet.

In [ ]:
# Find years with no African teams
all_mens_years = tournaments[
    tournaments['tournament_name'].str.contains("Men's")
]['year'].tolist()

africa_years = yearly['year'].tolist()
missing_years = [y for y in all_mens_years if y not in africa_years]

# Create rows for missing years with 0 teams
missing_df = pd.DataFrame({
    'year': missing_years,
    'African_Teams': 0,
    'Best_Performance': 0,
    'Best_Team': '',
    'Best_Performance_Label': 'No African teams'
})

# Combine and sort
yearly = pd.concat([yearly, missing_df]).sort_values('year').reset_index(drop=True)

# Add 2026 manually — 9 African teams is the historic expansion
row_2026 = pd.DataFrame([{
    'year': 2026,
    'African_Teams': 9,
    'Best_Performance': None,
    'Best_Team': 'Morocco, Nigeria, Senegal + 6 others',
    'Best_Performance_Label': 'Tournament in progress'
}])
yearly = pd.concat([yearly, row_2026]).reset_index(drop=True)

print(yearly.to_string())

## Step 7: Add Annotations

Key historical moments to label on the chart.

In [ ]:
# Annotation text for landmark moments
annotations = {
    1934: "Egypt: Africa's first WC appearance",
    1970: "Africa returns after 36 years",
    1990: "Cameroon reach quarter-finals",
    1998: "Expanded to 5 African slots",
    2010: "First WC on African soil",
    2022: "Morocco reach semi-finals",
    2026: "Historic: 9 African slots"
}

yearly['Annotation'] = yearly['year'].map(annotations).fillna('')

print(yearly[['year', 'African_Teams', 'Best_Team',
              'Best_Performance_Label', 'Annotation']].to_string())

## Step 8: Export to CSV

In [ ]:
# Export to Google Drive
# Change path to your own Google Drive path
yearly.to_csv('/content/drive/MyDrive/africa_worldcup_timeline.csv', index=False)
print('Saved successfully!')
print(f'Final shape: {yearly.shape}')